In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
import hashlib
import json
import shutil
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from scipy.integrate import trapezoid
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'src' / 'obp_utils.py').exists()), Path.cwd())
RAW = ROOT / 'data/raw/pitching'
OUT = ROOT / 'results/phase_decomposition'
FIG = ROOT / 'figures/phase_decomposition'
for folder in (RAW, OUT, FIG):
    folder.mkdir(parents=True, exist_ok=True)
AUTO_DOWNLOAD = True
OUTER_FOLDS = 5
INNER_FOLDS = 4
CV_SEEDS = (20261002, 20261003)
ALPHAS = (0.1, 1., 10., 100., 1000.)
BOOTSTRAPS = 2000
SEED = 20261002
SELECTED_PITCH = None
DURATION_LIMITS_MS = {'landing': 100., 'cocking': 300., 'acceleration': 80.}
PHASES = {'landing': ('fp_10_time', 'fp_100_time'),
          'cocking': ('fp_100_time', 'MER_time'),
          'acceleration': ('MER_time', 'BR_time'),
          'followthrough': ('BR_time', 'MIR_time')}
PRE_RELEASE = ('landing', 'cocking', 'acceleration')
EVENTS = ['fp_10_time', 'fp_100_time', 'MER_time', 'BR_time', 'MIR_time']
ANGLE_COLS = ['pelvis_angle_z', 'torso_angle_z', 'torso_angle_x', 'lead_knee_angle_x']
VELO_COLS = ['pelvis_velo_z', 'torso_velo_z', 'shoulder_velo_z', 'elbow_velo_x']
ENERGY_JOINTS = ('lead_knee', 'shoulder', 'elbow')
ENERGY_COLS = [j + '_energy_' + suffix for j in ENERGY_JOINTS
               for suffix in ('generated', 'transfer_stp', 'transfer_jfp')]
ARCHIVES = {
    'joint_angles': 'b4c4854ffcc52a0f757e0b3a3c86acdc4a2fc2e50516a76e62aabdb916afed46',
    'joint_velos': '5752f571b40acc4f02afc42dcfdb4b9eb191fcffd5fb25bc3c585e149941fb69',
    'energy_flow': '7732e35fbb3e53a2a6bc87b5118fcd6b7bc81496dffd0156f5d3561cc7854467'}
print('Delivery Phase Profiles & Pitch Velocity')
print('Fastballs only; phase importance means held-out predictive information, not causal contribution.')
print('First run downloads about 115 MB of missing archives; later runs reuse local files.')

In [ ]:
def sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as source:
        for block in iter(lambda: source.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

def download(url, path):
    if not AUTO_DOWNLOAD:
        raise FileNotFoundError(f'Missing {path}; set AUTO_DOWNLOAD=True or supply the official file.')
    print(f'Downloading {path.name} ...', flush=True)
    temporary = path.with_suffix(path.suffix + '.part')
    urlretrieve(url, temporary)
    temporary.replace(path)

base = 'https://github.com/drivelineresearch/openbiomechanics/releases/download/dataset-v1/'
csv_base = 'https://raw.githubusercontent.com/drivelineresearch/openbiomechanics/dataset-v1/baseball_pitching/data/'
for name, digest in ARCHIVES.items():
    target = RAW / (name + '.csv')
    if not target.exists():
        archive = RAW / ('pitching_' + name + '.zip')
        if not archive.exists():
            download(base + archive.name, archive)
        if sha256(archive) != digest:
            raise ValueError(f'{archive.name}: Dataset v1 archive hash mismatch')
        with zipfile.ZipFile(archive) as z:
            matches = [n for n in z.namelist() if Path(n).name == target.name]
            if len(matches) != 1:
                raise ValueError(f'Expected exactly one {target.name}')
            temporary = target.with_suffix('.csv.part')
            with z.open(matches[0]) as source, temporary.open('wb') as dest:
                shutil.copyfileobj(source, dest)
            temporary.replace(target)
for name, remote in [('metadata.csv', 'metadata.csv'), ('poi_metrics.csv', 'poi/poi_metrics.csv')]:
    if not (RAW / name).exists():
        download(csv_base + remote, RAW / name)
metadata = pd.read_csv(RAW / 'metadata.csv', dtype={'session_pitch': str, 'user': str})
poi = pd.read_csv(RAW / 'poi_metrics.csv', dtype={'session_pitch': str})
angles = pd.read_csv(RAW / 'joint_angles.csv', usecols=['session_pitch', 'time'] + EVENTS + ANGLE_COLS,
                     dtype={'session_pitch': str})
velos = pd.read_csv(RAW / 'joint_velos.csv', usecols=['session_pitch', 'time'] + EVENTS + VELO_COLS,
                    dtype={'session_pitch': str})
energy = pd.read_csv(RAW / 'energy_flow.csv', usecols=['session_pitch', 'time'] + EVENTS + ENERGY_COLS,
                     dtype={'session_pitch': str})
for name, table in [('metadata', metadata), ('POI', poi)]:
    if table.session_pitch.duplicated().any():
        raise ValueError(f'Duplicate pitch IDs in {name}')
for name, table in [('angles', angles), ('velos', velos), ('energy', energy)]:
    if table.duplicated(['session_pitch', 'time']).any():
        raise ValueError(f'Duplicate pitch/time pairs in {name}')
pitches = poi.loc[poi.pitch_type.eq('FF'), ['session_pitch', 'pitch_speed_mph']].merge(
    metadata[['session_pitch', 'user', 'session_mass_kg', 'session_height_m']],
    on='session_pitch', how='left', validate='one_to_one')
angle_groups = {k: g.sort_values('time') for k, g in angles.groupby('session_pitch', sort=False)}
velo_groups = {k: g.sort_values('time') for k, g in velos.groupby('session_pitch', sort=False)}
energy_groups = {k: g.sort_values('time') for k, g in energy.groupby('session_pitch', sort=False)}
print(f'Input: {len(pitches)} fastballs from {pitches.user.nunique()} pitchers')

In [ ]:
def phase_window(g, columns, lo, hi):
    if g is None or not np.isfinite([lo, hi]).all() or lo >= hi:
        raise ValueError('missing signal or invalid phase boundaries')
    t = g.time.to_numpy(float)
    if not np.isfinite(t).all() or np.any(np.diff(t) <= 0):
        raise ValueError('invalid time grid')
    if lo < t[0] or hi > t[-1]:
        raise ValueError('phase outside sampled recording')
    dt = np.median(np.diff(t))
    if not np.allclose(np.diff(t), dt, atol=0.00011, rtol=0.03):
        raise ValueError('irregular sampling')
    left = max(0, np.searchsorted(t, lo, side='right') - 1)
    right = min(len(t), np.searchsorted(t, hi, side='left') + 1)
    tt, values = t[left:right], g[columns].to_numpy(float)[left:right]
    if len(tt) < 2 or not np.isfinite(values).all():
        raise ValueError('missing phase samples or signal gaps')
    for i, col in enumerate(columns):
        if col in ('pelvis_angle_z', 'torso_angle_z'):
            values[:, i] = np.rad2deg(np.unwrap(np.deg2rad(values[:, i])))
    grid = np.r_[lo, tt[(tt > lo) & (tt < hi)], hi]
    sampled = np.column_stack([np.interp(grid, tt, values[:, i]) for i in range(len(columns))])
    return grid, sampled

def event_values(g):
    if g is None:
        raise ValueError('missing angle signal')
    values = {}
    for name in EVENTS:
        if g[name].nunique(dropna=False) != 1:
            raise ValueError('inconsistent event annotation')
        values[name] = float(g[name].iloc[0])
    return values

def extract_phase(ag, vg, eg, phase, events, mass):
    lo, hi = [events[name] for name in PHASES[phase]]
    ta, a = phase_window(ag, ANGLE_COLS, lo, hi)
    tv, v = phase_window(vg, VELO_COLS, lo, hi)
    f = {'duration_ms': 1000 * (hi - lo)}
    for i, col in enumerate(ANGLE_COLS):
        f[col + '_change_deg'] = a[-1, i] - a[0, i]
    for i, col in enumerate(VELO_COLS):
        f[col + '_mean_dps'] = trapezoid(v[:, i], tv) / (hi - lo)
        f[col + '_peak_dps'] = v[:, i].max()
    ef, energy_status = {}, 'available'
    try:
        te, e = phase_window(eg, ENERGY_COLS, lo, hi)
        for i, joint in enumerate(ENERGY_JOINTS):
            generated, stp, jfp = e[:, i * 3:i * 3 + 3].T
            ef[joint + '_generation_j_kg'] = trapezoid(np.maximum(generated, 0), te) / mass
            ef[joint + '_absorption_j_kg'] = trapezoid(np.maximum(-generated, 0), te) / mass
            ef[joint + '_transfer_stp_signed_j_kg'] = trapezoid(stp, te) / mass
            ef[joint + '_transfer_jfp_signed_j_kg'] = trapezoid(jfp, te) / mass
    except ValueError as exc:
        energy_status = str(exc)
        ef = {joint + '_' + kind: np.nan for joint in ENERGY_JOINTS for kind in
              ['generation_j_kg', 'absorption_j_kg', 'transfer_stp_signed_j_kg', 'transfer_jfp_signed_j_kg']}
    return f, ef, energy_status

records, audits, energy_records = [], [], []
for pitch in pitches.itertuples(index=False):
    key = pitch.session_pitch
    row = dict(session_pitch=key, pitcher_id=str(pitch.user), pitch_speed_mph=pitch.pitch_speed_mph,
               mass_kg=pitch.session_mass_kg, height_m=pitch.session_height_m)
    try:
        if pd.isna(pitch.user) or not np.isfinite([pitch.pitch_speed_mph, pitch.session_mass_kg,
                                                  pitch.session_height_m]).all() or pitch.session_mass_kg <= 0 or pitch.session_height_m <= 0:
            raise ValueError('missing outcome, pitcher ID, or body size')
        ag, vg, eg = angle_groups.get(key), velo_groups.get(key), energy_groups.get(key)
        events = event_values(ag)
        required = [events[n] for n in EVENTS[:4]]
        if not np.isfinite(required).all() or np.any(np.diff(required) <= 0):
            raise ValueError('missing or unordered pre-release events')
        for other in (vg, eg):
            if other is not None:
                for name in EVENTS[:4]:
                    if other[name].nunique(dropna=False) != 1 or not np.isclose(
                            float(other[name].iloc[0]), events[name], atol=.0001):
                        raise ValueError('event disagreement across signal tables')
        for phase in PRE_RELEASE:
            f, ef, es = extract_phase(ag, vg, eg, phase, events, pitch.session_mass_kg)
            row.update({phase + '__' + k: value for k, value in {**f, **ef}.items()})
            energy_records.append(dict(session_pitch=key, pitcher_id=str(pitch.user), phase=phase,
                                       energy_status=es, **ef))
        row.update(events)
        row['long_phase_flag'] = any(row[p + '__duration_ms'] > limit
                                     for p, limit in DURATION_LIMITS_MS.items())
        records.append(row)
        status = 'retained'
    except (ValueError, KeyError, TypeError) as exc:
        status = str(exc)
    audits.append(dict(session_pitch=key, status=status))
audit = pd.DataFrame(audits)
features = pd.DataFrame(records)
if features.empty:
    display(audit.status.value_counts())
    raise ValueError('No fastballs retained; inspect the event/signal audit')
valid_ids = set(features.session_pitch)
energy_summary = pd.DataFrame(energy_records)
energy_summary = energy_summary[energy_summary.session_pitch.isin(valid_ids)].copy()
followthrough = []
for row in features.itertuples(index=False):
    key = row.session_pitch
    try:
        f, ef, status = extract_phase(angle_groups[key], velo_groups[key], energy_groups.get(key),
                                      'followthrough', event_values(angle_groups[key]), row.mass_kg)
        followthrough.append(dict(session_pitch=key, pitcher_id=row.pitcher_id, phase='followthrough',
                                  status='retained', energy_status=status, **f, **ef))
    except ValueError as exc:
        followthrough.append(dict(session_pitch=key, pitcher_id=row.pitcher_id, phase='followthrough', status=str(exc)))
followthrough = pd.DataFrame(followthrough)
audit.to_csv(OUT / 'extraction_audit.csv', index=False)
features.to_csv(OUT / 'pitch_phase_features.csv', index=False)
energy_summary.to_csv(OUT / 'phase_energy.csv', index=False)
followthrough.to_csv(OUT / 'followthrough_descriptive.csv', index=False)
display(audit.status.value_counts().rename('pitches').to_frame())
print(f'Retained {len(features)} pitches from {features.pitcher_id.nunique()} pitchers')
print(f'Long-phase sensitivity flags: {int(features.long_phase_flag.sum())} pitches')
print('Mean and peak angular velocities use OBP directional conventions; angle changes remain signed.')
print('Energy generation/absorption are positive magnitudes; transfer channels remain signed and separate.')

In [ ]:
def pitcher_weights(ids):
    ids = pd.Series(np.asarray(ids))
    return (1 / ids.map(ids.value_counts())).to_numpy(float)

def group_splits(ids, folds, seed):
    ids = np.asarray(ids)
    unique = np.unique(ids)
    if len(unique) < folds:
        raise ValueError('Too few pitchers for grouped validation')
    shuffled = np.random.default_rng(seed).permutation(unique)
    for held_out in np.array_split(shuffled, folds):
        test = np.flatnonzero(np.isin(ids, held_out))
        train = np.flatnonzero(~np.isin(ids, held_out))
        if set(ids[train]) & set(ids[test]):
            raise AssertionError('Pitcher leakage')
        yield train, test

def group_mae(y, prediction, ids):
    return pd.DataFrame({'id': ids, 'error': np.abs(np.asarray(y) - prediction)}).groupby('id').error.mean().mean()

def fit_ridge(x, y, ids, alpha):
    model = make_pipeline(StandardScaler(), Ridge(alpha=alpha))
    model.fit(x, y, ridge__sample_weight=pitcher_weights(ids))
    return model

def tune_alpha(x, y, ids, seed):
    splits = list(group_splits(ids, INNER_FOLDS, seed))
    scores = []
    for alpha in ALPHAS:
        predicted = np.full(len(y), np.nan)
        for train, test in splits:
            model = fit_ridge(x.iloc[train], y[train], ids[train], alpha)
            predicted[test] = model.predict(x.iloc[test])
        scores.append(group_mae(y, predicted, ids))
    return ALPHAS[int(np.argmin(scores))]

kin_columns = {p: [c for c in features if c.startswith(p + '__') and not c.endswith('j_kg')]
               for p in PRE_RELEASE}
energy_columns = {p: [c for c in features if c.startswith(p + '__') and c.endswith('j_kg')]
                  for p in PRE_RELEASE}
body = ['height_m', 'mass_kg']
all_kin = sum(kin_columns.values(), [])
all_energy = sum(energy_columns.values(), [])
if any('followthrough' in c for c in all_kin + all_energy):
    raise AssertionError('Post-release feature in velocity model')
main_specs = {'training_mean': [], 'body_size': body,
              **{p + '_only': body + kin_columns[p] for p in PRE_RELEASE},
              'all_phases': body + all_kin,
              **{'omit_' + p: body + sum([kin_columns[q] for q in PRE_RELEASE if q != p], [])
                 for p in PRE_RELEASE}}
energy_specs = {'training_mean': [], 'body_size': body, 'all_phases': body + all_kin,
                'all_phases_plus_energy': body + all_kin + all_energy,
                **{'omit_block_' + p: body + sum([kin_columns[q] + energy_columns[q]
                    for q in PRE_RELEASE if q != p], []) for p in PRE_RELEASE}}

predictions, tuning, fold_audit = [], [], []
for cohort, table, specs in [('kinematic', features, main_specs),
                            ('energy_complete', features.dropna(subset=all_energy), energy_specs),
                            ('duration_sensitivity', features[~features.long_phase_flag].dropna(subset=all_energy),
                             {**main_specs, 'all_phases_plus_energy': body + all_kin + all_energy})]:
    table = table.reset_index(drop=True)
    if table.pitcher_id.nunique() < max(20, OUTER_FOLDS + INNER_FOLDS):
        print(f'{cohort}: too few complete pitchers; skipped')
        continue
    x, y, ids = table, table.pitch_speed_mph.to_numpy(float), table.pitcher_id.to_numpy()
    for repeat, seed in enumerate(CV_SEEDS):
        for fold, (train, test) in enumerate(group_splits(ids, OUTER_FOLDS, seed)):
            fold_audit.append(dict(cohort=cohort, repeat=repeat, fold=fold,
                                   training_pitchers='|'.join(sorted(set(ids[train]))),
                                   test_pitchers='|'.join(sorted(set(ids[test])))))
            for label, cols in specs.items():
                if cols:
                    alpha = tune_alpha(x[cols].iloc[train], y[train], ids[train], seed + fold)
                    model = fit_ridge(x[cols].iloc[train], y[train], ids[train], alpha)
                    pred = model.predict(x[cols].iloc[test])
                else:
                    alpha = np.nan
                    pred = np.repeat(np.average(y[train], weights=pitcher_weights(ids[train])), len(test))
                tuning.append(dict(cohort=cohort, repeat=repeat, fold=fold, model=label, alpha=alpha))
                for j, index in enumerate(test):
                    predictions.append(dict(cohort=cohort, repeat=repeat, fold=fold, model=label,
                                             session_pitch=table.session_pitch.iloc[index], pitcher_id=ids[index],
                                             actual_mph=y[index], predicted_mph=pred[j]))
        print(f'{cohort}: completed repeat {repeat + 1}/{len(CV_SEEDS)}', flush=True)
predictions = pd.DataFrame(predictions)
if predictions.empty:
    raise ValueError('No validation predictions; inspect complete-pitcher counts')
predictions.to_csv(OUT / 'held_out_predictions.csv', index=False)
pd.DataFrame(tuning).to_csv(OUT / 'ridge_tuning.csv', index=False)
pd.DataFrame(fold_audit).to_csv(OUT / 'validation_fold_audit.csv', index=False)

In [ ]:
def interval(values):
    values = np.asarray(values, dtype=float)
    rng = np.random.default_rng(SEED)
    draws = values[rng.integers(0, len(values), size=(BOOTSTRAPS, len(values)))].mean(axis=1)
    return values.mean(), *np.quantile(draws, [.025, .975])

predictions['absolute_error'] = abs(predictions.actual_mph - predictions.predicted_mph)
metric_rows = []
for (cohort, label), g in predictions.groupby(['cohort', 'model']):
    per_pitcher = g.groupby('pitcher_id').absolute_error.mean()
    mean, low, high = interval(per_pitcher)
    weights = pitcher_weights(g.pitcher_id)
    metric_rows.append(dict(cohort=cohort, model=label, pitcher_weighted_mae_mph=mean,
                            bootstrap_ci_low=low, bootstrap_ci_high=high,
                            pitcher_weighted_rmse_mph=np.sqrt(np.average(
                                (g.actual_mph - g.predicted_mph) ** 2, weights=weights)),
                            n_pitchers=g.pitcher_id.nunique(), n_pitches=g.session_pitch.nunique()))
metrics = pd.DataFrame(metric_rows)
metrics.to_csv(OUT / 'model_metrics.csv', index=False)
comparison_rows = []
for cohort, g in predictions.groupby('cohort'):
    errors = g.groupby(['pitcher_id', 'model']).absolute_error.mean().unstack('model')
    contrasts = [(p, 'omit_' + p, 'all_phases') for p in PRE_RELEASE] if cohort != 'energy_complete' else [
        ('added_energy', 'all_phases', 'all_phases_plus_energy')] + [
        (p + '_kinematics_and_energy', 'omit_block_' + p, 'all_phases_plus_energy') for p in PRE_RELEASE]
    for label, reduced, full in contrasts:
        if reduced not in errors or full not in errors:
            continue
        delta = errors[reduced] - errors[full]
        estimate, lo, hi = interval(delta.dropna())
        comparison_rows.append(dict(cohort=cohort, comparison=label, reduced_model=reduced, full_model=full,
                                    mae_increase_when_removed_mph=estimate, bootstrap_ci_low=lo,
                                    bootstrap_ci_high=hi))
comparisons = pd.DataFrame(comparison_rows)
repeat_scores = predictions.groupby(['cohort', 'repeat', 'model', 'pitcher_id']).absolute_error.mean().groupby(
    ['cohort', 'repeat', 'model']).mean().rename('mae_mph').reset_index()
repeat_scores.to_csv(OUT / 'repeat_model_metrics.csv', index=False)
comparisons.to_csv(OUT / 'phase_ablation.csv', index=False)
display(metrics.round(3))
display(comparisons.round(3))
print('Positive ablation values mean removing those features worsened held-out MAE.')
print('Negative values mean the smaller model predicted better; phase importance is not a share of ball speed.')
print('Bootstrap intervals resample pitchers from fixed outer predictions; they do not refit the whole CV procedure.')
print('Repeated-fold predictions are not treated as independent observations.')

In [ ]:
energy_check = []
for row in features.itertuples(index=False):
    g = energy_groups.get(row.session_pitch)
    for joint in ('shoulder', 'elbow'):
        try:
            t, values = phase_window(g, [joint + '_energy_generated'], row.fp_100_time, row.BR_time)
            generation = trapezoid(np.maximum(values[:, 0], 0), t)
            absorption = trapezoid(np.maximum(-values[:, 0], 0), t)
            official = poi.set_index('session_pitch').loc[row.session_pitch]
            energy_check.append(dict(session_pitch=row.session_pitch, joint=joint,
                                      generation_j=generation, absorption_j=absorption,
                                      official_generation_j=official[joint + '_generation_fp_br'],
                                      official_absorption_j=official[joint + '_absorption_fp_br']))
        except (ValueError, KeyError):
            continue
energy_check = pd.DataFrame(energy_check)
energy_check.to_csv(OUT / 'energy_poi_crosscheck.csv', index=False)
if not energy_check.empty:
    energy_check['generation_abs_diff_j'] = abs(energy_check.generation_j - energy_check.official_generation_j)
    energy_check['absorption_abs_diff_j'] = abs(energy_check.absorption_j - energy_check.official_absorption_j)
    display(energy_check.groupby('joint')[['generation_abs_diff_j', 'absorption_abs_diff_j']].median().round(3))
print('Integrals use exact interpolated phase boundaries. Official POI summaries may use different sampling rules.')
print('Generation/absorption estimates are descriptive. Joint transfer channels are not added into a whole-body total.')

In [ ]:
selected = str(SELECTED_PITCH) if SELECTED_PITCH is not None else features.session_pitch.iloc[0]
if selected not in set(features.session_pitch):
    raise KeyError(f'Pitch {selected} was not retained')
row = features.set_index('session_pitch').loc[selected]
g = velo_groups[selected]
fig, ax = plt.subplots(figsize=(11, 5))
colors = {'landing': '#e4ae48', 'cocking': '#467ca8', 'acceleration': '#be5946', 'followthrough': '#609d78'}
for phase in PRE_RELEASE:
    lo, hi = [row[name] for name in PHASES[phase]]
    ax.axvspan(1000 * (lo - row.fp_100_time), 1000 * (hi - row.fp_100_time),
               alpha=.14, color=colors[phase], label=phase.title())
for col, label in [('pelvis_velo_z', 'Pelvis'), ('torso_velo_z', 'Torso'), ('shoulder_velo_z', 'Shoulder IR')]:
    mask = g.time.between(row.fp_10_time, row.BR_time)
    ax.plot(1000 * (g.loc[mask, 'time'] - row.fp_100_time), g.loc[mask, col], label=label)
for name in EVENTS[:4]:
    ax.axvline(1000 * (row[name] - row.fp_100_time), color='gray', linestyle=':', linewidth=1)
ax.set(xlabel='Time from foot plant (ms)', ylabel='Directional angular velocity (deg/s)',
       title=f'Pitch {selected}: pre-release phases, {row.pitch_speed_mph:.1f} mph')
ax.legend(ncol=3)
fig.tight_layout(); fig.savefig(FIG / 'phase_timeline.png', dpi=180); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sample = metrics[metrics.cohort.eq('kinematic')].sort_values('pitcher_weighted_mae_mph')
axes[0].barh(sample.model, sample.pitcher_weighted_mae_mph, color='#467ca8')
axes[0].set(xlabel='Held-out MAE (mph; lower is better)', title='Models on the same kinematic cohort')
contrast = comparisons[comparisons.cohort.eq('kinematic')]
if len(contrast):
    delta = contrast.mae_increase_when_removed_mph.to_numpy()
    axes[1].errorbar(delta, np.arange(len(contrast)),
                     xerr=np.maximum(0, np.vstack([delta - contrast.bootstrap_ci_low,
                                                 contrast.bootstrap_ci_high - delta])), fmt='o', color='#be5946')
    axes[1].set_yticks(np.arange(len(contrast)), contrast.comparison)
    axes[1].axvline(0, color='gray', linestyle=':')
axes[1].set(xlabel='MAE increase when phase removed (mph)', title='Additional phase information; bootstrap 95% intervals')
fig.tight_layout(); fig.savefig(FIG / 'phase_model_comparison.png', dpi=180); plt.show()

summary = energy_summary.groupby(['pitcher_id', 'phase'])[[j + '_generation_j_kg' for j in ENERGY_JOINTS]].mean()
summary = summary.groupby('phase').mean().reindex(PRE_RELEASE)
fig, ax = plt.subplots(figsize=(9, 5))
summary.rename(columns={j + '_generation_j_kg': j.replace('_', ' ').title() for j in ENERGY_JOINTS}).plot.bar(ax=ax)
ax.set(xlabel='Phase', ylabel='Estimated positive generation (J/kg)',
       title='Pitcher-weighted joint generation, shown separately')
ax.tick_params(axis='x', rotation=0)
fig.tight_layout(); fig.savefig(FIG / 'phase_energy_profile.png', dpi=180); plt.show()

profile = features.groupby('pitcher_id')[['pitch_speed_mph', 'mass_kg', 'height_m'] + all_kin + all_energy].mean()
profile['n_pitches'] = features.groupby('pitcher_id').size()
profile.to_csv(OUT / 'pitcher_profiles.csv')
profile.rank(pct=True).mul(100).drop(columns='n_pitches').to_csv(OUT / 'pitcher_profile_sample_percentiles.csv')
manifest = dict(dataset='OpenBiomechanics Dataset v1', fastballs_only=True,
                retained_pitches=len(features), retained_pitchers=int(features.pitcher_id.nunique()),
                outer_folds=OUTER_FOLDS, inner_folds=INNER_FOLDS, cv_seeds=list(CV_SEEDS),
                ridge_alphas=list(ALPHAS), bootstrap_resamples=BOOTSTRAPS,
                phase_boundaries=PHASES, long_phase_sensitivity_limits_ms=DURATION_LIMITS_MS,
                archive_sha256=ARCHIVES,
                input_sha256={f: sha256(RAW / f) for f in ['metadata.csv', 'poi_metrics.csv',
                                 'joint_angles.csv', 'joint_velos.csv', 'energy_flow.csv']},
                limitations=['Observational prediction; no causal contribution percentages.',
                             'All model preprocessing and alpha selection are inside training folds.',
                             'Follow-through is descriptive only and excluded from velocity models.',
                             'Energy signals are power in watts integrated over real seconds.',
                             'Transfer channels remain separate; no whole-body energy budget.',
                             'Intervals condition on fitted cross-validation models.',
                             'Small dataset; individual coaching interventions are not validated.'])
(OUT / 'run_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print(f'Tables and individual profiles: {OUT}')
print(f'Figures: {FIG}')